 ### Model 1
 - 1 row = pitcher vs hitter
 - target = did hitter strike out?   # 0 or 1
 - model = XGBoost classifier
 - output = probability hitter strikes out  

 - sum hitter probabilities across lineup = expected pitcher strikeouts

 - 'y' = hitter_strikeOuts

- Step 1
    - Load data from mlb.dbo.fact_hitter_pitcher_matchup_model_features
- Step 2
    - Choose the target column
- Step 3
    - Drop leakage / non-model columns
- Step 4
    - Handle nulls and data types
- Step 5
    - Train/test split
- Step 6
    - Train first XGBoost classifier
- Step 7
    - Evaluate:
        - accuracy
        - precision / recall
        - ROC-AUC
        - log loss
        - calibration check
- Step 8
    - Generate hitter strikeout probabilities
Step 9
- Aggregate probabilities to pitcher expected Ks

Did this hitter strike out against this pitcher in this game?
    0 = no strikeout
    1 = at least one strikeout

1. Imports
2. Connect to SQL Server
3. Load model table
4. Inspect data
5. Build target
6. Drop leakage columns
7. Prepare X and y
8. Train/test split
9. Train XGBoost classifier
10. Evaluate model
11. Predict probabilities
12. Aggregate to pitcher expected Ks

In [28]:
import pandas as pd
from sqlalchemy import create_engine

SERVER = "localhost"
DATABASE = "mlb"
DRIVER = "ODBC Driver 17 for SQL Server"

connection_string = (
    f"mssql+pyodbc://@{SERVER}/{DATABASE}"
    f"?driver={DRIVER.replace(' ', '+')}"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string)

query = """
SELECT *
FROM mlb.dbo.fact_hitter_pitcher_matchup_model_features
"""

df = pd.read_sql(query, engine)

print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nFirst 20 columns:")
print(df.columns[:20].tolist())

Shape: (62220, 543)

First 5 rows:


,gamePk,game_date,season,hitter_id,hitter_name,hitter_position,hitter_team_id,hitter_team_name,pitcher_id,pitcher_name,...,lineup_weighted_whiff_rate_vs_rhp_last_3,lineup_weighted_whiff_rate_vs_rhp_last_5,lineup_weighted_whiff_rate_vs_rhp_last_10,lineup_weighted_whiff_rate_vs_lhp_last_3,lineup_weighted_whiff_rate_vs_lhp_last_5,lineup_weighted_whiff_rate_vs_lhp_last_10,lineup_num_high_k_hitters,lineup_num_power_hitters,lineup_num_high_whiff_hitters,lineup_num_strong_bats
0,776680,2025-08-19,2025,647304,Josh Naylor,1B,136,Seattle Mariners,650911,Cristopher Sánchez,...,0.107142,0.102312,0.107361,0.089465,0.113731,0.103993,4,2,1,3
1,776680,2025-08-19,2025,668227,Randy Arozarena,LF,136,Seattle Mariners,661395,Jhoan Duran,...,0.107142,0.102312,0.107361,0.089465,0.113731,0.103993,4,2,1,3
2,776680,2025-08-19,2025,668227,Randy Arozarena,LF,136,Seattle Mariners,650911,Cristopher Sánchez,...,0.107142,0.102312,0.107361,0.089465,0.113731,0.103993,4,2,1,3
3,776680,2025-08-19,2025,670042,Luke Raley,RF,136,Seattle Mariners,661395,Jhoan Duran,...,0.107142,0.102312,0.107361,0.089465,0.113731,0.103993,4,2,1,3
4,776680,2025-08-19,2025,553993,Eugenio Suárez,3B,136,Seattle Mariners,650911,Cristopher Sánchez,...,0.107142,0.102312,0.107361,0.089465,0.113731,0.103993,4,2,1,3



First 20 columns:
['gamePk', 'game_date', 'season', 'hitter_id', 'hitter_name', 'hitter_position', 'hitter_team_id', 'hitter_team_name', 'pitcher_id', 'pitcher_name', 'pitcher_team_id', 'pitcher_team_name', 'pitcher_throws', 'hitter_stand', 'hitter_strikeOuts', 'pitches_seen_vs_pitcher', 'swings_vs_pitcher', 'whiffs_vs_pitcher', 'called_strikes_vs_pitcher', 'matchup_whiff_rate']


In [29]:
# Convert hitter_strikeOuts to binary target
df["target_hitter_k"] = (df["hitter_strikeOuts"] > 0).astype(int)

print(df["target_hitter_k"].value_counts())
print(df["target_hitter_k"].value_counts(normalize=True))

target_hitter_k
0    42120
1    20100
Name: count, dtype: int64
target_hitter_k
0    0.676953
1    0.323047
Name: proportion, dtype: float64


In [30]:
# Define leakage columns - Ensure we clean it by ensuring that the model does not see actual results which is cheating.
leakage_cols = [
    "hitter_strikeOuts",        # source of target
    "pitcher_strikeOuts",       # future info

    # matchup outcomes (these happen during the game)
    "pitches_seen_vs_pitcher",
    "swings_vs_pitcher",
    "whiffs_vs_pitcher",
    "called_strikes_vs_pitcher",

    # derived from above (VERY important to drop)
    "matchup_whiff_rate",
    "matchup_called_strike_rate",
    "matchup_csw_rate"
]

In [31]:
drop_cols = [
    "gamePk",
    "game_date",
    "hitter_name",
    "pitcher_name",
    "hitter_team_name",
    "pitcher_team_name"
]

In [32]:
# new variable to combine leakage_cols and drop_cols to drop them
df_model = df.drop(columns=leakage_cols + drop_cols, errors="ignore")

print("Old shape:", df.shape)
print("New shape:", df_model.shape)

Old shape: (62220, 544)
New shape: (62220, 529)


In [33]:
# [] result to ensure they are gone. 

[col for col in leakage_cols if col in df_model.columns]

[]

In [34]:
# drop target_hitter_k which is the new variable created from strikeOut to binary
# creating y as the predict 
# to confirm x has all model features and y is the target
X = df_model.drop(columns=["target_hitter_k"])      # The features
y = df_model["target_hitter_k"]                     # Target

print("X shape:", X.shape)
print("Y shape:", y.shape)
print("\nTarget check:")
print(y.value_counts(normalize=True))

X shape: (62220, 528)
Y shape: (62220,)

Target check:
target_hitter_k
0    0.676953
1    0.323047
Name: proportion, dtype: float64


In [35]:
# split the data
# stratify=y removed and train 2025 and test 2026
from sklearn.model_selection import train_test_split

train_df = df_model[df_model["season"] == 2025]
test_df  = df_model[df_model["season"] == 2026]

X_train = train_df.drop(columns=["target_hitter_k"])
y_train = train_df["target_hitter_k"]

X_test = test_df.drop(columns=["target_hitter_k"])
y_test = test_df["target_hitter_k"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

Train shape: (47346, 528)
Test shape: (14874, 528)

Train target distribution:
target_hitter_k
0    0.673953
1    0.326047
Name: proportion, dtype: float64

Test target distribution:
target_hitter_k
0    0.6865
1    0.3135
Name: proportion, dtype: float64


In [36]:
# checking the remaining column types
print(X.dtypes.value_counts())

float64    508
int64       15
str          5
Name: count, dtype: int64


In [37]:
object_cols = X.select_dtypes(include=["object"]).columns.tolist()
print("Number of object columns:", len(object_cols))
print(object_cols[:50])

Number of object columns: 5
['hitter_position', 'pitcher_throws', 'hitter_stand', 'hitter_lineup_position', 'hitter_lineup_position_name']


C:\Users\andre\AppData\Local\Temp\ipykernel_19596\896964950.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = X.select_dtypes(include=["object"]).columns.tolist()


In [38]:
from sklearn.preprocessing import LabelEncoder

X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

label_encoders = {}

for col in object_cols:
    le = LabelEncoder()

    X_train_encoded[col] = le.fit_transform(X_train_encoded[col].astype(str))

    X_test_encoded[col] = X_test_encoded[col].map(
        lambda s: le.transform([str(s)])[0] if str(s) in le.classes_ else -1
    )

    label_encoders[col] = le

print("Encoding complete ✅")
print(X_train_encoded.dtypes.value_counts())
print(X_test_encoded.dtypes.value_counts())

Encoding complete ✅
float64    508
int64       20
Name: count, dtype: int64
float64    508
int64       20
Name: count, dtype: int64


In [40]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train_encoded, y_train)

print("Model trained ✅")

Model trained ✅


In [42]:
# Results
y_pred = model.predict(X_test_encoded)
y_prob = model.predict_proba(X_test_encoded)[:, 1]

from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("Log Loss:", log_loss(y_test, y_prob))

Accuracy: 1.0
ROC AUC: 1.0
Log Loss: 0.00031791416287629457
